# Quantization-Aware Training for Spyx SNNs

This tutorial wires Google's [`qwix`](https://github.com/google/qwix) JAX quantization library into Spyx so you can train a spiking network with int8 weights/activations end-to-end.

**Required extras:**

```bash
uv pip install "spyx[quant]"  # pulls qwix from GitHub via tool.uv.sources
```

If you're not on `uv`:

```bash
pip install "git+https://github.com/google/qwix"
```

## Why a wrapper?

`qwix.quantize_model` happily quantizes any Flax NNX module, but on a spiking network the right defaults are non-obvious:

- The dense `nnx.Linear` / `nnx.Conv` layers benefit from int8 weights + activations.
- The spiking dynamics (`LIF`, `CuBaLIF`, `ALIF`, `IF`) and the leaky readout (`LI`) are *very* sensitive to integer rounding because they recurse on the membrane potential. Quantizing them collapses the network to silence.

`spyx.quant.linear_only_rules()` encodes this default: regex matches `Linear` and `Conv` modules, leaves the neurons alone.

In [ ]:
import jax
import jax.numpy as jnp
import optax
from flax import nnx

import spyx
import spyx.nn as snn

assert spyx.quant.available(), (
    "qwix is not installed. Install with `uv pip install \"spyx[quant]\"`."
)

## Build a small SNN

We use a tiny network for demonstration; the same pattern scales to the full SHD trainer in `docs/examples/surrogate_gradient/SurrogateGradientTutorial.ipynb`.

In [ ]:
IN_DIM, HIDDEN, N_CLASSES = 32, 64, 10
BATCH = 8

def make_model(seed=0):
    rngs = nnx.Rngs(seed)
    return snn.Sequential(
        nnx.Linear(IN_DIM, HIDDEN, use_bias=False, rngs=rngs),
        snn.LIF((HIDDEN,), activation=spyx.axn.triangular(), rngs=rngs),
        nnx.Linear(HIDDEN, HIDDEN, use_bias=False, rngs=rngs),
        snn.LIF((HIDDEN,), activation=spyx.axn.triangular(), rngs=rngs),
        nnx.Linear(HIDDEN, N_CLASSES, use_bias=False, rngs=rngs),
        snn.LI((N_CLASSES,), rngs=rngs),
    )

fp_model = make_model()
sample_x = jnp.ones((BATCH, IN_DIM))
sample_state = fp_model.initial_state(BATCH)
fp_out, _ = fp_model(sample_x, sample_state)
print("fp32 output shape:", fp_out.shape)

## Wrap with `spyx.quant.quantize`

`quantize` traces the model with the example inputs (so qwix can discover the modules), then returns a new `nnx.Module` whose Linear layers are wrapped in `qwix.QArray` with `int8` weights and activations. The default mode is `qat` (quantization-aware training); pass `mode="ptq"` for post-training quantization.

In [ ]:
qmodel = spyx.quant.quantize(fp_model, sample_x, sample_state)
q_out, _ = qmodel(sample_x, sample_state)
print("quantized output shape:", q_out.shape)
print("max abs diff vs fp32:", float(jnp.max(jnp.abs(q_out - fp_out))))

## Train through the quantized model

The QAT model fits straight into the standard NNX training loop: `nnx.Optimizer(qmodel, optax.lion(...), wrt=nnx.Param)` plus `nnx.value_and_grad`. The straight-through estimator inside qwix passes gradients through the quant/dequant pair, so the loss is fully differentiable.

In [ ]:
Loss = spyx.fn.integral_crossentropy()
optimizer = nnx.Optimizer(qmodel, optax.lion(3e-4), wrt=nnx.Param)

@nnx.jit
def train_step(model, optimizer, x, state, targets):
    def loss_fn(m):
        out, _ = m(x, state)
        # treat the readout as a trace integrated over a single timestep
        traces = out[:, None, :]
        return Loss(traces, targets)
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

key = jax.random.PRNGKey(0)
for step in range(20):
    key, k = jax.random.split(key)
    x = jax.random.normal(k, (BATCH, IN_DIM))
    targets = jax.random.randint(k, (BATCH,), 0, N_CLASSES)
    loss = float(train_step(qmodel, optimizer, x, sample_state, targets))
    if step % 4 == 0:
        print(f"step {step:2d}: loss={loss:.4f}")

## Customising the quantization rules

`spyx.quant.linear_only_rules` is a thin shorthand. Tweak the dtypes, or use `qwix.QuantizationRule` directly for finer control.

In [ ]:
import qwix

# int4 weights, fp32 activations - good for memory-bound deployment.
rules = spyx.quant.weights_only_rules(weight_qtype="int4")
qmodel_int4 = spyx.quant.quantize(make_model(), sample_x, sample_state, rules=rules)
out_int4, _ = qmodel_int4(sample_x, sample_state)
print("int4 weights output shape:", out_int4.shape)

# Custom: only quantize the second hidden Linear, leave the readout in fp32.
custom_rules = [
    qwix.QuantizationRule(
        module_path=r".*layers/2.*",  # Sequential places layers under .layers/<idx>
        weight_qtype="int8",
        act_qtype="int8",
    ),
]
qmodel_partial = spyx.quant.quantize(make_model(), sample_x, sample_state, rules=custom_rules)
out_partial, _ = qmodel_partial(sample_x, sample_state)
print("partial-quant output shape:", out_partial.shape)

## Notes

- For full QAT runs on real datasets, plug `spyx.quant.quantize` into any of the SHD tutorials right after model construction. The training loop is unchanged.
- Qwix supports `int1`-`int8`, `nf4`, and `fp8`. The native fast paths are `int4`, `int8`, and `fp8` on TPU/H100; everything else is emulated.
- For pure mixed-precision (bf16/fp16) without integer quantization, set `jax.config.update("jax_default_matmul_precision", "bfloat16")` instead - qwix is overkill for that case.